# Imports

In [ ]:
import numpy as np
import cv2
import PIL
import mediapipe.python.solutions.face_detection as mp_face_detection
import torch
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from torchvision.models import ResNet50_Weights
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from tqdm import tqdm

# Emotion Detection

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
])
preprocess_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    preprocess
])

In [ ]:
train_dataset = ImageFolder(root='DATASET/train', transform=preprocess_aug)

train_size = int(len(train_dataset.samples) * 0.75)
val_size = len(train_dataset.samples) - train_size

train_dataset, val_dataset = random_split(train_dataset, lengths=[train_size, val_size])

test_dataset = ImageFolder(root='DATASET/test', transform=preprocess)

In [ ]:
BATCH_SIZE = 32

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=4, shuffle=True, pin_memory=True, persistent_workers=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True, persistent_workers=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True, persistent_workers=True)

In [ ]:
def train_model(model, train_dataloader, val_dataloader, test_dataloader, criterion, optimizer, scheduler, patience, epochs):
    train_acc_list, train_loss_list, val_acc_list, val_loss_list = [], [], [], []
    best_val_loss = float('inf')
    patience_counter = 0
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        model.train()
        pbar = tqdm(train_dataloader, desc=f'Epoch: {epoch+1} / {epochs}')
        sum_training_loss = 0
        sum_training_correct_pred = 0
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device).long().view(-1)

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                output = model(images)
                loss = criterion(output, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            sum_training_loss += loss.item() * images.size(0)
            sum_training_correct_pred += (output.argmax(axis=1) == labels).sum().item()
            pbar.set_postfix(acc=f'{(sum_training_correct_pred / len(train_dataset)) * 100:.2f}%', loss=f'{sum_training_loss / len(train_dataset):.4f}')

        train_loss = sum_training_loss / len(train_dataset)
        train_acc = sum_training_correct_pred / len(train_dataset)
        train_loss_list.append(train_loss)
        train_acc_list.append(train_acc * 100)

        model.eval()
        sum_validation_loss = 0
        sum_validation_correct_pred = 0
        pbar = tqdm(val_dataloader, desc='Validation')
        with torch.no_grad():
            for images, labels in pbar:
                images = images.to(device)
                labels = labels.to(device).long().view(-1)
                with torch.amp.autocast('cuda'):
                    output = model(images)
                    loss = criterion(output, labels)
                sum_validation_loss += loss.item() * images.size(0)
                sum_validation_correct_pred += (output.argmax(axis=1) == labels).sum().item()
                pbar.set_postfix(acc=f'{(sum_validation_correct_pred / len(val_dataset)) * 100:.2f}%', loss=f'{sum_validation_loss / len(val_dataset):.4f}')

        val_loss = sum_validation_loss / len(val_dataset)
        val_acc = sum_validation_correct_pred / len(val_dataset)
        val_acc_list.append(val_acc * 100)
        val_loss_list.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pt')
            patience_counter = 0
            print(f'Epoch {epoch+1}: new best val loss {best_val_loss:.4f}, model saved.')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}')
                break

    return train_acc_list, train_loss_list, val_acc_list, val_loss_list

In [ ]:
model = models.resnet50(weights='IMAGENET1K_V1')
weights = ResNet50_Weights.DEFAULT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_features = model.fc.in_features

model.fc = torch.nn.Sequential(
    torch.nn.Dropout(0.2),
    torch.nn.Linear(in_features, 7),
)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = torch.nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='min', factor=0.1, patience=3)

In [ ]:
# Set TRAIN = True to train (creates best_model.pt).
# Set EVALUATE = True to print a test-set report (needs best_model.pt).
# Leave both False to go straight to the webcam demo.
TRAIN = False
EVALUATE = False


In [ ]:
if TRAIN:
    EPOCHS = 20
    PATIENCE = 5

    train_acc_list, train_loss_list, valid_acc_list, valid_loss_list = train_model(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        scheduler = scheduler,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        test_dataloader=test_dataloader,
        patience = PATIENCE,
        epochs=EPOCHS
    )
    plt.subplot(2, 1, 1)
    plt.title("Loss")
    plt.plot(train_loss_list, label="train")
    plt.plot(valid_loss_list, label="valid")
    plt.grid()
    plt.legend()

    plt.subplot(2, 1, 2)
    plt.title("Acc")
    plt.plot(train_acc_list, label="train")
    plt.plot(valid_acc_list, label="valid")
    plt.grid()
    plt.legend()

    plt.show()

In [ ]:
if EVALUATE:
    model.load_state_dict(torch.load('best_model.pt'))

    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for landmark, label in test_dataloader:
            landmark = landmark.to(device)
            label = label.to(device).long().view(-1)
            output = model(landmark)
            preds = output.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label.cpu().numpy())

    print(classification_report(all_labels, all_preds))

# Web Cam

In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_emotion(prediction):
    prediction = torch.nn.functional.softmax(prediction, dim=1).cpu().detach().numpy()
    top = np.argmax(prediction, axis=1)
    score = float(np.amax(prediction))
    result = Labels[top[0]]
    return result, score

def preprocess_webcam(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = PIL.Image.fromarray(image)
    image = data_transforms(image)
    image = image.float()
    image = image.to(device)
    image = image.unsqueeze(0)
    return image

In [ ]:
cam = cv2.VideoCapture(0)
show_res = 'Nothing'
show_score = 0.0
box_coords = None

Labels = {
    0: 'Surprise', 1: 'Fear', 2: 'Disgust', 3: 'Happiness',
    4: 'Sadness', 5: 'Anger', 6: 'Neutral'
}

model.load_state_dict(torch.load("best_model.pt", map_location="cpu"))
model = model.to(device)
model.eval()

frame_count = 0
with mp_face_detection.FaceDetection(
    model_selection=0,
    min_detection_confidence=0.5
) as face_detection:
    with torch.no_grad():
        while cam.isOpened():
            ret, frame = cam.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_count += 1
            height, width, _ = frame.shape

            faces = face_detection.process(rgb)
            num_faces = len(faces.detections or [])

            cv2.putText(
                frame,
                f"Faces: {num_faces}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (100, 0, 180),
                2,
            )
            if frame_count % 5 == 0: 
                if faces.detections:
                    bbox = faces.detections[0].location_data.relative_bounding_box 
                    x, y = int(bbox.xmin * width), int(bbox.ymin * height) # Mediapipe uses fractions so we have to multiply by our webcam's height and width to see where it is exactly
                    bw, bh = int(bbox.width * width), int(bbox.height * height)
                    box_coords = (max(0,x), max(0,y), x+bw, y+bh) # Since some part of our face might be outside the frame, Mediapipe might give us a negative number, the max() function is to prevent that
                    image = frame[max(0, y):y+bh, max(0, x):x+bw] # Sliced
                    if image.size > 0:
                        image_data = preprocess_webcam(image)
                        prediction = model(image_data)
                        result, score = predict_emotion(prediction)

                        if score >= 0.75:
                            show_res = result
                            show_score = score
                        else:
                            show_res = 'Uncertain'
                            show_score = score

                else:
                    show_res = 'No face detected'
                    show_score = 0.0
            if box_coords:
                cv2.rectangle(frame, (box_coords[0], box_coords[1]), (box_coords[2], box_coords[3]), (255, 0, 0), 2)
            cv2.putText(frame, f"{show_res} ({show_score*100:.1f}%)", (220, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (100, 0, 100), 2)

            cv2.imshow('Emotion Detection', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
cam.release()
cv2.destroyAllWindows()